# Garson-bot — Hey Garson Wake Word Model Training
## A. Title And Warning

**Amaç:** openWakeWord otomatik eğitim pipeline'ı ile `hey_garson.onnx` modeli üret.

> ⚠️ **Colab GPU gereklidir.** Menü: *Çalışma Zamanı → Çalışma zamanı türünü değiştir → T4 GPU*  
> ⚠️ Bu notebook'u her seferinde **baştan çalıştır** — hücreler arası bağımlılık var.  
> ⚠️ Eğitim ~30-60 dk sürer (T4). Tarayıcı sekmesini kapatma.

**Sonuç:** `hey_garson.onnx` dosyasını indir →  
`robot_waiter_ai/models/hey_garson.onnx` olarak kaydet.


## B. Runtime Check

In [ ]:
# GPU ve RAM kontrolü
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        "GPU bulunamadı!\n"
        "Menü: Çalışma Zamanı → Çalışma zamanı türünü değiştir → T4 GPU"
    )

gpu_info = result.stdout.strip()
print(f"✅ GPU: {gpu_info}")
print(f"Python: {sys.version}")

import os
mem_bytes = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
print(f"RAM: {mem_bytes / 1024**3:.1f} GB")


## C. Project Setup

In [ ]:
import os
from pathlib import Path

# openWakeWord reposunu klonla (eğitim araçları için)
if not Path("openWakeWord").exists():
    !git clone --depth 1 https://github.com/dscripka/openWakeWord.git
    print("✅ openWakeWord klonlandı")
else:
    print("✅ openWakeWord zaten mevcut")

os.chdir("openWakeWord")
print(f"Çalışma dizini: {os.getcwd()}")


## D. Install Dependencies

In [ ]:
# openWakeWord + eğitim bağımlılıkları
!pip install -q -e ".[train]"

# Meta MMS-TTS (Türkçe, offline, transformers tabanlı — Piper yerine)
!pip install -q transformers>=4.40 scipy

# Ses işleme araçları
!pip install -q audiomentations soundfile librosa

# Doğrulama
import importlib
for pkg in ["openwakeword", "transformers", "audiomentations", "torch"]:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', '?')
        print(f'  ✅ {pkg} {ver}')
    except ImportError:
        print(f'  ❌ {pkg} — kurulum başarısız!')


## E. Wake Word Configuration

In [ ]:
import json
from pathlib import Path

# ── Hedef wake word ──────────────────────────────────────────────────────────
TARGET_PHRASE = "hey garson"

# Telaffuz varyasyonları — model daha robust olur
PHRASE_VARIATIONS = [
    "hey garson",
    "heyy garson",
    "hey garsoon",
    "hay garson",
    "hey garsan",
    "hey garsın",    # ı → ın varyasyonu
]

# ── TTS motoru ───────────────────────────────────────────────────────────────
# Meta MMS-TTS: offline, 16kHz, Turkish, no binary required
MMS_MODEL_ID = "facebook/mms-tts-tur"

# ── Eğitim parametreleri ─────────────────────────────────────────────────────
N_POSITIVE_SAMPLES   = 5_000   # her varyasyon başına TTS örnekleri
N_NEGATIVE_SAMPLES   = 50_000  # arka plan
VALIDATION_SPLIT     = 0.10
BATCH_SIZE           = 64
EPOCHS               = 50
LEARNING_RATE        = 1e-3
MODEL_NAME           = "hey_garson"
OUTPUT_DIR           = Path("/content/hey_garson_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'TARGET_PHRASE     : {TARGET_PHRASE}')
print(f'PHRASE_VARIATIONS : {len(PHRASE_VARIATIONS)} adet')
print(f'TTS motoru        : {MMS_MODEL_ID}')
print(f'N_POSITIVE_SAMPLES: {N_POSITIVE_SAMPLES}')
print(f'N_NEGATIVE_SAMPLES: {N_NEGATIVE_SAMPLES}')
print(f'OUTPUT_DIR        : {OUTPUT_DIR}')


## F. Generate Positive Samples

In [ ]:
# Meta MMS-TTS ile Türkçe wake word ses örnekleri üret
# facebook/mms-tts-tur — 16kHz mono WAV, offline, transformers

import random
import numpy as np
import scipy.io.wavfile
import soundfile as sf
import torch
from pathlib import Path
from transformers import VitsModel, AutoTokenizer
from audiomentations import Compose, TimeStretch, PitchShift, AddGaussianNoise

POSITIVE_DIR = OUTPUT_DIR / "positive_clips"
POSITIVE_DIR.mkdir(exist_ok=True)

# ── Model yükle (ilk çalıştırmada HuggingFace'den indirilir, ~200 MB) ────────
print(f"MMS-TTS modeli yükleniyor: {MMS_MODEL_ID}")
_tokenizer = AutoTokenizer.from_pretrained(MMS_MODEL_ID)
_tts_model = VitsModel.from_pretrained(MMS_MODEL_ID)
_tts_model.eval()
_SAMPLE_RATE = _tts_model.config.sampling_rate   # 16000
print(f"✅ MMS-TTS hazır — sample_rate={_SAMPLE_RATE} Hz\n")

# Ses augmentation pipeline
augment = Compose([
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.6),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.4),
])


def synthesize_mms(text: str, out_path: Path) -> bool:
    """MMS-TTS ile metni WAV dosyasına yaz."""
    try:
        inputs = _tokenizer(text, return_tensors="pt")
        with torch.no_grad():
            waveform = _tts_model(**inputs).waveform.squeeze().cpu().numpy()
        pcm = (waveform * 32767).clip(-32768, 32767).astype(np.int16)
        scipy.io.wavfile.write(str(out_path), rate=_SAMPLE_RATE, data=pcm)
        return out_path.exists() and out_path.stat().st_size > 0
    except Exception as exc:
        print(f"  ❌ Sentez hatası ({text!r}): {exc}")
        return False


total = 0
samples_per_variation = N_POSITIVE_SAMPLES // len(PHRASE_VARIATIONS)

print(f"Üretilecek: {len(PHRASE_VARIATIONS)} varyasyon × "
      f"~{samples_per_variation} örnek = ~{N_POSITIVE_SAMPLES} pozitif")

for variation in PHRASE_VARIATIONS:
    var_slug = variation.replace(" ", "_")
    print(f"\n  → '{variation}'")

    for i in range(samples_per_variation):
        wav_path = POSITIVE_DIR / f"{var_slug}_{i:05d}.wav"

        if not synthesize_mms(variation, wav_path):
            continue

        if random.random() < 0.7:
            audio, sr = sf.read(str(wav_path))
            audio_aug = augment(audio.astype(np.float32), sample_rate=sr)
            aug_path = POSITIVE_DIR / f"{var_slug}_{i:05d}_aug.wav"
            sf.write(str(aug_path), audio_aug, sr)

        total += 1
        if total % 500 == 0:
            print(f"    {total} örnek üretildi…")

print(f"\n✅ Toplam pozitif örnek: {total}")
print(f"   Dizin: {POSITIVE_DIR}")


## G. Download Negative Data

In [ ]:
# Negatif örnekler: FMA müzik + MUSAN gürültü + CommonVoice TR (diğer kelimeler)
# openWakeWord bu veri setlerini otomatik indirebilir — veya manuel path ver

from openwakeword.data import download_background_noise
from pathlib import Path

NEGATIVE_DIR = OUTPUT_DIR / "negative_clips"
NEGATIVE_DIR.mkdir(exist_ok=True)

print("Arka plan gürültüsü indiriliyor (MUSAN + FMA subset)…")
print("Bu işlem ~5-10 dk sürebilir.")

try:
    # openWakeWord'ün yerleşik indirici
    download_background_noise(
        output_dir=str(NEGATIVE_DIR),
        n_samples=N_NEGATIVE_SAMPLES,
    )
    neg_count = len(list(NEGATIVE_DIR.rglob("*.wav")))
    print(f"\n✅ Negatif örnek: {neg_count} WAV dosyası")
except Exception as e:
    # Manuel fallback: MUSAN'ı direkt indir
    print(f"Otomatik indirme hatası: {e}")
    print("MUSAN manuel indiriliyor…")
    !wget -q --show-progress -O /tmp/musan.tar.gz \
        https://www.openslr.org/resources/17/musan.tar.gz
    !tar -xf /tmp/musan.tar.gz -C {NEGATIVE_DIR}
    neg_count = len(list(NEGATIVE_DIR.rglob("*.wav")))
    print(f"✅ MUSAN kuruldu: {neg_count} dosya")

# CommonVoice Türkçe — "garson" olmayan kelimeler (opsiyonel ama tavsiye edilir)
print("\nNOT: CommonVoice TR için HuggingFace token gerekebilir.")
print("      Atlamak istersen bu hücreyi yorum satırına al.")


## H. Verify Training Data

In [ ]:
# Veri setini doğrula — sayılar, süre, format

import soundfile as sf
import numpy as np
from pathlib import Path

def audit_clips(directory: Path, label: str, max_check: int = 200):
    files = list(directory.rglob("*.wav"))
    if not files:
        print(f"  ⚠️ {label}: Dosya bulunamadı — {directory}")
        return

    durations = []
    errors = 0
    for f in files[:max_check]:
        try:
            info = sf.info(str(f))
            if info.samplerate != 16000:
                # Yeniden örnekle
                audio, sr = sf.read(str(f))
                import librosa
                audio_r = librosa.resample(audio.astype(np.float32),
                                           orig_sr=sr, target_sr=16000)
                sf.write(str(f), audio_r, 16000)
            durations.append(sf.info(str(f)).duration)
        except Exception:
            errors += 1

    total = len(files)
    med_dur = float(np.median(durations)) if durations else 0
    print(f"  {label}: {total:,} dosya | medyan süre: {med_dur:.2f}s | "
          f"hata: {errors}")

print("Veri seti denetimi:")
audit_clips(POSITIVE_DIR, "Pozitif (hey garson)")
audit_clips(NEGATIVE_DIR, "Negatif (arka plan)")

# Oran kontrolü
pos = len(list(POSITIVE_DIR.rglob("*.wav")))
neg = len(list(NEGATIVE_DIR.rglob("*.wav")))
ratio = neg / pos if pos > 0 else 0
print(f"\nNegatif/Pozitif oranı: {ratio:.1f}x (ideal: 5-15x)")
if ratio < 3:
    print("  ⚠️ Negatif örnek az — daha fazla indirmeyi düşün")
elif ratio > 30:
    print("  ⚠️ Negatif örnek çok fazla — eğitim uzun sürer")
else:
    print("  ✅ Oran uygun")

assert pos >= 1000, f"Çok az pozitif örnek: {pos}"
assert neg >= 5000, f"Çok az negatif örnek: {neg}"
print("\n✅ Veri seti hazır — eğitime devam edebilirsin")


## I. Training Cell

In [ ]:
# openWakeWord model eğitimi
# GPU'da ~30-60 dk (T4), CPU'da ~3-5 saat

import torch
from openwakeword.train import train_model
from pathlib import Path
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {device}")
if device == "cpu":
    print("⚠️ GPU bulunamadı — eğitim çok yavaş olacak!")

print(f"\nEğitim başlıyor: {MODEL_NAME}")
print(f"  Epoch: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")
print(f"  Pozitif: {len(list(POSITIVE_DIR.rglob('*.wav'))):,}")
print(f"  Negatif: {len(list(NEGATIVE_DIR.rglob('*.wav'))):,}")
print("-" * 50)

t_start = time.time()

try:
    results = train_model(
        model_name=MODEL_NAME,
        positive_clips_dir=str(POSITIVE_DIR),
        negative_clips_dir=str(NEGATIVE_DIR),
        output_dir=str(OUTPUT_DIR),
        val_split=VALIDATION_SPLIT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        device=device,
        target_false_positive_rate=config["false_positive_rate"],
    )
    elapsed = (time.time() - t_start) / 60
    print(f"\n✅ Eğitim tamamlandı ({elapsed:.1f} dk)")
    print(f"   Sonuçlar: {results}")
except Exception as e:
    print(f"\n❌ Eğitim hatası: {e}")
    raise


## J. Evaluate & Test

In [ ]:
# Model performansını değerlendir

import openwakeword
import numpy as np
from pathlib import Path

# Eğitilmiş modeli yükle
onnx_candidates = list(OUTPUT_DIR.rglob("*.onnx"))
if not onnx_candidates:
    raise FileNotFoundError(f"ONNX dosyası bulunamadı: {OUTPUT_DIR}")

onnx_path = onnx_candidates[0]
print(f"Model: {onnx_path}")

# openWakeWord ile yükle
from openwakeword.model import Model
oww = Model(wakeword_models=[str(onnx_path)], inference_framework="onnx")

# Pozitif örnekler üzerinde test
print("\nPositive set test (ilk 100 örnek):")
pos_files = list(POSITIVE_DIR.rglob("*.wav"))[:100]
tp = 0
for f in pos_files:
    import soundfile as sf
    audio, _ = sf.read(str(f), dtype="int16")
    preds = oww.predict(audio)
    score = max(preds.values()) if preds else 0.0
    if score >= 0.5:
        tp += 1

tpr = tp / len(pos_files) * 100 if pos_files else 0
print(f"  True Positive Rate : {tpr:.1f}%  ({tp}/{len(pos_files)})")
print(f"  Hedef              : ≥ 85%")

# Negatif örnekler üzerinde test (ilk 200 örnek)
print("\nNegative set test (ilk 200 örnek):")
neg_files = list(NEGATIVE_DIR.rglob("*.wav"))[:200]
fp = 0
for f in neg_files:
    import soundfile as sf
    try:
        audio, _ = sf.read(str(f), dtype="int16")
        # 1280 sample'lık chunk'larla dene
        for i in range(0, min(len(audio), 16000*3), 1280):
            chunk = audio[i:i+1280]
            if len(chunk) < 1280:
                break
            preds = oww.predict(chunk)
            score = max(preds.values()) if preds else 0.0
            if score >= 0.5:
                fp += 1
                break
    except Exception:
        pass

fpr = fp / len(neg_files) * 100 if neg_files else 0
print(f"  False Positive Rate: {fpr:.1f}%  ({fp}/{len(neg_files)})")
print(f"  Hedef              : ≤ 5%")

print("\n" + ("✅ Model kabul edilebilir seviyede"
               if tpr >= 85 and fpr <= 5
               else "⚠️ Model hedefin altında — epoch artır veya veri ekle"))


## K. Export & Download

In [ ]:
# ONNX modeli doğrula ve indir

from google.colab import files
from pathlib import Path
import shutil

# ONNX dosyasını bul
onnx_files = list(OUTPUT_DIR.rglob("*.onnx"))
if not onnx_files:
    raise FileNotFoundError("ONNX dosyası bulunamadı!")

onnx_src = onnx_files[0]
onnx_dest = Path(f"/content/{MODEL_NAME}.onnx")
shutil.copy2(onnx_src, onnx_dest)

size_kb = onnx_dest.stat().st_size / 1024
print(f"✅ Model: {onnx_dest.name}  ({size_kb:.1f} KB)")

# ONNX runtime ile son doğrulama
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(str(onnx_dest))
input_name = sess.get_inputs()[0].name
input_shape = sess.get_inputs()[0].shape
print(f"   Input: {input_name}  shape={input_shape}")

dummy = np.zeros([1, 96], dtype=np.float32)   # openWakeWord embedding boyutu
try:
    out = sess.run(None, {input_name: dummy})
    print(f"   Output shape: {out[0].shape}")
    print("✅ ONNX runtime doğrulaması başarılı")
except Exception as e:
    print(f"⚠️ ONNX doğrulama hatası: {e}")

print(f"\nDosya indiriliyor: {onnx_dest.name}")
print("İndirme tamamlandıktan sonra şu dizine koy:")
print("  robot_waiter_ai/models/hey_garson.onnx")
print()

# Colab üzerinden indir
files.download(str(onnx_dest))
print("\n✅ İşlem tamamlandı!")
print("   Sonraki adım: python -m robot_waiter_ai.speech.mic (donanım smoke-test)")
